# 📅 2025-01-XX (X요일) 개발 노트 : 인프라 구축 및 데이터 로드

## 🎯 오늘의 목표
- [x] Docker 인프라(PostgreSQL+pgvector, Redis) 정상 구동 확인
- [x] Python 3.12 정식 설치 및 가상환경 구축
- [x] Django 마이그레이션 완료 (games, users, game_metrics 테이블 생성)
- [x] FastAPI 서버 정상 구동 확인
- [ ] GPT-5.4 데이터 4,190개 로드 (진행 중)

## 🛠 진행 상황 및 핵심 기록

### 아키텍처 확정
- **Django (포트 8001):** 데이터 관리, Admin 페이지, 데이터 파이프라인, DB 마이그레이션 전담
- **FastAPI (포트 8000):** AI 검색 엔진, pgvector 활용 Two-Tower ANN 검색, Redis 캐싱, 실시간 API
- **PostgreSQL:** pgvector 확장 활성화, HNSW 인덱스 (m=16, ef_construction=64)
- **Redis:** Semantic Cache, Rate Limiting, Django↔FastAPI Pub/Sub 동기화

### DB 스키마 (Single Source of Truth = Django)
- games - 게임 기본 정보 + AI 텍스트 필드 (marketing_hook, one_line_summary 등)
- game_metrics - 52개 지표 (33 수치 + 9 태그) + pure_embedding (33차원 벡터)
- users - CustomUser (Django AbstractUser 확장)


### 핵심 설계 결정
1. **pure_embedding:** 가중치 없는 정규화 벡터 (DB 저장)
2. **검색 시 가중치:** 유저 쿼리에 개인화 가중치 적용 (실시간)
3. **Bayesian Smoothing:** 리뷰 0개 게임도 공정 평가 (prior_mean=0.75, prior_strength=10)
4. **비대칭 페널티:** MUST_LOW는 지수, MUST_HIGH는 제곱

## 🚨 트러블슈팅 (문제 및 해결)

### 1. Pydantic `extra_forbidden` 에러
- **문제:** FastAPI 실행 시 `pydantic_core._pydantic_core.ValidationError: extra_forbidden`
- **원인:** Pydantic v2의 Settings에서 extra 필드 처리 방식 변경
- **해결:** `config.py`에서 `model_config = SettingsConfigDict(extra="ignore")` 추가

### 2. `ModuleNotFoundError: sync_service`
- **문제:** FastAPI 시작 시 `services.sync_service` 모듈 없음
- **원인:** `sync_service.py` 파일 누락
- **해결:** Redis Pub/Sub 핸들러가 포함된 `sync_service.py` 생성

### 3. `load_games.py` 모델 불일치 에러
- **문제:** `TypeError: GameMetric() got unexpected keyword arguments 'marketing_hook', 'one_line_summary'...`
- **원인:** AI 텍스트 필드가 `Game` 모델에 있는데 `GameMetric` 생성 시 전달
- **해결:** `load_games.py` 전면 재작성 - Game/GameMetric 필드 정확히 분리, pure_embedding 자동 생성 포함

### 4. Python 3.14 / 3.12 충돌
- **문제:** `py` 명령이 3.14를 실행, Store 버전 제거 불가
- **해결:** 정식 Python 3.12 설치 후 직접 경로로 가상환경 생성
  ```bash
  "/c/Program Files/Python312/python.exe" -m venv .venv
